# Generate synthetic dataset

In [6]:
import pandas as pd
import numpy as np
import fakeitmakeit as fm
import pathlib

data_dir = pathlib.Path('./')

## Presentations

In [7]:
n = 300
n_supervisors = 100
n_moderators = 6

supervisors = pd.Series(list(set(fm.name() for _ in range(n_supervisors))))
moderators = supervisors.sample(n_moderators, replace=False).to_numpy()

s1_name = supervisors.sample(n, replace=True).to_numpy()
s2_name = supervisors.sample(n, replace=True).to_numpy()


def who_is_the_moderator(row):
    if row.s1_name in moderators:
        return row.s1_name
    if row.s2_name in moderators:
        return row.s2_name
    return pd.Series(moderators).sample(1, replace=True).iat[0]


for i, p in enumerate(s2_name):
    while p == s1_name[i]:
        p = supervisors.sample(1, replace=True).to_numpy()[0]
    s2_name[i] = p

presentations = (
    fm.cohort(n)
    .assign(
        student=pd.col("first_name") + " " + pd.col("last_name"),
        s1_name=s1_name,
        s2_name=s2_name,
        moderator=lambda df: df.apply(who_is_the_moderator, axis=1),
    )
    .drop_duplicates(subset=["student"])
    .loc[:, ["student", "s1_name", "s2_name", "moderator"]]
    .rename_axis("id")
)

assert presentations.student.is_unique
assert presentations.s1_name.eq(presentations.s2_name).sum() == 0
assert presentations.student.eq(presentations.s1_name).sum() == 0
assert presentations.student.eq(presentations.s2_name).sum() == 0

print(f"Number of presentations: {len(presentations)}")
print(f"Number of unique s1_name: {len(presentations.s1_name.unique())}")
print(f"Number of unique s2_name: {len(presentations.s2_name.unique())}")
print (f"Number of unique moderators: {len(presentations.moderator.unique())}")

presentations.to_csv(data_dir / "presentations.csv", index=True)

presentations.sample(20)

Number of presentations: 292
Number of unique s1_name: 97
Number of unique s2_name: 92
Number of unique moderators: 6


,student,s1_name,s2_name,moderator
id,,,,
yiy81,Yang Yan,Brett Rodriguez,Charles Dunn,Andre Phillips
pq49,Ping Qiu,Johnny Scott,Robert Fowler,Mark Garcia
js680,Jie Sun,Kristy Elliott,Douglas Adams,Mark Garcia
my75,Ming Ye,Louis Oliver,Timothy Chapman,Andre Phillips
yh54,Yong Huang,Jillian Long,William Richards,William Richards
mr850,Mark Robinson,Kevin Young,William Flores,Timothy Stout
veh10,Veronica Hill,Cassandra Mccormick,Jonathan Lam,Cheryl Stevens
gog599,Guiying Gong,Justin White,Stephanie Ward,Charles Baker
csl2024,Chao Lu,Sarah Hayes,Angelica Fletcher,Mark Garcia


## Session start times

In [8]:
session_start_times = pd.DataFrame(
    {
        "session": [1, 2, 3, 4, 5, 6, 7, 8],
        "start_time": [
            "09:30",
            "10:15",
            "11:00",
            "11:45",
            "14:00",
            "14:45",
            "15:30",
            "16:15",
        ],
    }
).astype({"start_time": "datetime64[ns]"}).set_index("session").loc[:, "start_time"].dt.time

session_start_times.to_csv('session-start-times.csv', index=True)

session_start_times

session
1    09:30:00
2    10:15:00
3    11:00:00
4    11:45:00
5    14:00:00
6    14:45:00
7    15:30:00
8    16:15:00
Name: start_time, dtype: object

## Unavailability table

In [9]:
# Availability exceptions table — one row per unavailability rule.
#
# Semantics of "day" and "session" (both nullable, using pandas' Int64):
#   - day = <value>, session = NaN      -> person unavailable ALL DAY on that day
#   - day = NaN,      session = <value> -> person unavailable during that session, EVERY day
#   - day = <value>,  session = <value> -> person unavailable for that specific (day, session) slot only
#   - day = NaN,      session = NaN     -> person unavailable for the ENTIRE conference
#
# NaN acts as a wildcard meaning "applies to all values" for that column.

n_restrictions = 60

# Allow repeated people so one person can have multiple restrictions.
people = supervisors.sample(n_restrictions, replace=True).to_numpy()

# Build nullable day/session vectors with mixed rule types (day-only, session-only, specific-slot, global).
rule_types = np.random.choice(
    ["day_only", "session_only", "specific_slot", "global"],
    size=n_restrictions,
    p=[0.30, 0.30, 0.35, 0.05],
)

days = np.full(n_restrictions, np.nan)
sessions = np.full(n_restrictions, np.nan)

for i, rule in enumerate(rule_types):
    if rule == "day_only":
        days[i] = np.random.randint(1, 8)
    elif rule == "session_only":
        sessions[i] = np.random.randint(1, 9)
    elif rule == "specific_slot":
        days[i] = np.random.randint(1, 8)
        sessions[i] = np.random.randint(1, 9)

unavailable = pd.DataFrame(
    {
        "person": people,
        "day": days,
        "session": sessions,
    }
).astype({"day": "Int64", "session": "Int64"})

print(f"Number of restrictions: {len(unavailable)}")
print(f"People with multiple restrictions: {(unavailable.person.value_counts() > 1).sum()}")

unavailable.to_csv("unavailable.csv", index=False)

unavailable.sample(5)

Number of restrictions: 60
People with multiple restrictions: 11


,person,day,session
4,Sara Welch,1,<NA>
37,William Flores,<NA>,<NA>
46,Amy Mckinney,3,4
23,Bethany Hicks,<NA>,6
2,Audrey Howard,5,<NA>
